# StepForge Flow v1.5 — training on Colab

Trains both networks on a multi-source corpus built with
`tools/beatforge/adapters/stepmania/neural/synth` and exports the ONNX model the
desktop app loads. Seven cells; every one can be re-run after a disconnect and
picks up where it stopped (checkpoints are written to Drive after every epoch).

**Before you start**, on the machine that holds the song packs:

```bash
pip install -r requirements.txt -r requirements-train.txt -r requirements-synth.txt
M=tools/beatforge/adapters/stepmania/neural
PYTHONPATH=tools python3 -m beatforge.adapters.stepmania.neural.synth.stems_cache   --packs-dir <packs> --out-dir build/stems_cache --only-manifest <locked manifest.json>
PYTHONPATH=tools python3 -m beatforge.adapters.stepmania.neural.train.corpus        --packs-dir <packs> --stems-dir build/stems_cache --split-manifest <locked manifest.json> --out-dir build/v15_corpus/real
PYTHONPATH=tools python3 -m beatforge.adapters.stepmania.neural.synth.audio_augment --real-dir build/v15_corpus/real --stems-dir build/stems_cache --out-dir build/v15_corpus/aug_stem --require-rubberband
PYTHONPATH=tools python3 -m beatforge.adapters.stepmania.neural.synth.groove_synth  --out-dir build/v15_corpus/synth_groove --num-tracks 300 --seed 20261119
PYTHONPATH=tools python3 -m beatforge.adapters.stepmania.neural.synth.chart_thinning --real-dir build/v15_corpus/real --out-dir build/v15_corpus/synth_thin
PYTHONPATH=tools python3 -m beatforge.adapters.stepmania.neural.synth.style_patterns --out-dir build/v15_corpus/synth_style --total-charts 2400 --seed 20261119
PYTHONPATH=tools python3 -m beatforge.adapters.stepmania.neural.synth.audio_grounder --corpus-root build/v15_corpus
PYTHONPATH=tools python3 -m beatforge.adapters.stepmania.neural.synth.colab_bundle   --corpus-root build/v15_corpus --out-dir build/colab_upload
```

and copy `build/colab_upload/` to `MyDrive/StepForge/v15_upload/`.

Validation and checkpoint selection use the `val` split of `real/` only. No
synthetic or augmented item can reach it: the loader refuses a corpus where one
does, and so does the bundle check in cell 2.

## 1 · Drive, code, GPU

In [ ]:
REPO   = "https://github.com/Sweet-Papa-Technologies/StepForge.git"
BRANCH = "main"
DRIVE  = "/content/drive/MyDrive/StepForge"
UPLOAD = f"{DRIVE}/v15_upload"        # tarballs from colab_bundle
RUNS   = f"{DRIVE}/v15_runs"          # checkpoints + logs (survive a disconnect)
CORPUS = "/content/v15_corpus"        # local SSD: never train from Drive directly

import os, subprocess, sys
from google.colab import drive
drive.mount("/content/drive")
os.makedirs(RUNS, exist_ok=True)

if not os.path.isdir("/content/StepForge/.git"):
    subprocess.run(["git", "clone", "--depth", "1", "-b", BRANCH, REPO, "/content/StepForge"], check=True)
else:
    subprocess.run(["git", "-C", "/content/StepForge", "pull", "--ff-only"], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "/content/StepForge/requirements.txt",
                "-r", "/content/StepForge/requirements-train.txt", "onnxruntime"], check=True)
os.environ["PYTHONPATH"] = "/content/StepForge/tools"
os.environ["PYTHONUNBUFFERED"] = "1"

import torch
assert torch.cuda.is_available(), "no GPU: Runtime > Change runtime type > A100 or L4"
name = torch.cuda.get_device_name(0)
AMP = "bf16" if torch.cuda.is_bf16_supported() else "fp16"
print(name, "| mixed precision:", AMP)
NEURAL = "beatforge.adapters.stepmania.neural"

def run(module, *args):
    """Run one of the package's command-line tools and stream its output."""
    cmd = [sys.executable, "-W", "ignore", "-m", f"{NEURAL}.{module}", *map(str, args)]
    print("$", " ".join(cmd[4:]), flush=True)
    subprocess.run(cmd, check=True)

## 2 · Unpack the corpus onto the local SSD

Verifies every tarball's checksum and the split rules.

In [ ]:
import json
if not os.path.exists(f"{CORPUS}/real/manifest.json"):
    run("synth.colab_bundle", "unpack", "--bundle-dir", UPLOAD, "--corpus-root", CORPUS)
for src in sorted(os.listdir(CORPUS)):
    m = json.load(open(f"{CORPUS}/{src}/manifest.json"))
    print(f"{src:22s} {m['n_songs']:5d} shards  {m['n_charts_kept']:6d} charts  {m['splits']}")

## 3 · Stage A — RhythmNet

Phase 1 (65 epochs): human charts 35 %, acoustic variants 30 %, synthesized
grooves 20 %, thinned low-tier charts 15 %. Phase 2 (30 epochs): human charts
55 %, stem-balance remixes 30 %, thinned 10 %, grooves 5 %. The best
checkpoint is chosen on real validation songs, without stem channels (the way
the app runs): `0.50 AP + 0.25 F1 on Hard/Challenge 16ths + 0.25 beat-phase accuracy`.
About 25 minutes on an A100.

In [ ]:
run("train.train_stage_a", "--corpus-root", CORPUS, "--checkpoint-dir", f"{RUNS}/stage_a",
    "--phase1-epochs", 65, "--phase2-epochs", 30, "--batch", 32, "--amp", AMP,
    "--dropout", 0.25, "--weight-decay", 0.03, "--p-drop-stems", 0.50,
    "--sub16-weight", 1.45, "--sub12-weight", 1.30,
    "--phase-margin-weight", 0.10, "--phase-shift-prob", 0.25, "--workers", 8)

## 4 · Seat the step patterns on audio

Sixty percent of the programmatic step patterns are laid over windows of
training audio at a matching tempo; the rest stay audio-free. Needs no trained
model (the step network has its own audio encoder), so this is skipped if the
bundle already carries `synth_style_grounded/`.

In [ ]:
if not os.path.exists(f"{CORPUS}/synth_style_grounded/manifest.json"):
    run("synth.audio_grounder", "--corpus-root", CORPUS)

## 5 · Stage B — ChoreoNet

Phase 1 (100 epochs): human charts 55 % (audio withheld on 20 % of them),
grounded patterns 30 %, audio-free patterns 15 %. Phase 2 (45 epochs): human
charts 75 % (10 % withheld), grounded 20 %, audio-free 5 %. Chosen on real
validation cross-entropy; the style probe is logged every 5 epochs.
About 35 minutes on an A100.

In [ ]:
run("train.train_stage_b", "--corpus-root", CORPUS, "--checkpoint-dir", f"{RUNS}/stage_b",
    "--phase1-epochs", 100, "--phase2-epochs", 45, "--batch", 64, "--amp", AMP,
    "--p-drop-audio", 0.20, "--p-drop-audio2", 0.10, "--workers", 8)

## 6 · Export, parity check, phrase index

Writes `stage_a_rhythm.onnx`, `stage_b_choreo.onnx` and their int8 variants,
and refuses to write `model_meta.json` unless both graphs match PyTorch to
1e-4 and the step network's training-only "no audio" vector is absent from
the exported graph. The inputs and outputs are the ones v1.4 has.

In [ ]:
MODEL = "/content/stepforge_flow_v15"
run("train.export_onnx", "--stage-a", f"{RUNS}/stage_a/best.pt", "--stage-b", f"{RUNS}/stage_b/best.pt",
    "--out", MODEL, "--int8", "--trained-with-stems", "--version", "1.5")
run("style_rag", "--corpus", f"{CORPUS}/real", "--out", MODEL)
meta = json.load(open(f"{MODEL}/model_meta.json"))
print("parity:", meta["stage_a"]["parity_max_abs_diff"], meta["stage_b"]["parity_max_abs_diff"])
print("stage A val:", {k: round(v, 4) for k, v in meta["metrics"]["stage_a"].items() if k in ("ap", "r_prec", "f1_hard16", "phase_acc", "val_score")})
print("stage B val:", {k: round(v, 4) for k, v in meta["metrics"]["stage_b"].items() if k in ("ce", "acc", "gen_hard", "gen_dstep")})

## 7 · Package the release candidate

Weights, training logs and the manifests that were trained on, zipped to
Drive. Download it into `models/stepforge_flow_v15/`, then calibrate the
decoder on the tuning songs (`calibrate_decoder.py`) and run the benchmark
before it replaces the shipped model.

In [ ]:
import shutil
os.makedirs(f"{MODEL}/train", exist_ok=True)
for stage in "ab":
    for f in ("log.jsonl", "config.json"):
        shutil.copy(f"{RUNS}/stage_{stage}/{f}", f"{MODEL}/train/stage_{stage}_{f}")
for src in sorted(os.listdir(CORPUS)):
    shutil.copy(f"{CORPUS}/{src}/manifest.json", f"{MODEL}/train/manifest_{src}.json")
out = shutil.make_archive(f"{DRIVE}/v15_release_candidate", "zip", MODEL)
print(out, round(os.path.getsize(out) / 1e6), "MB")